# Claude API cost optimization

Baseline vs. optimized weekly cost for the routes this repo's agents call through — `triangulation-orchestrator`, WKB `check`/`bridge`/`dispatch` workflows — using the same pricing table as `scripts/usage_tracker.py` (PR #9). Three independent optimization levers are quantified against a synthetic one-week usage log shaped like what `UsageTracker.record()` would see per call: **prompt caching**, **model right-sizing**, and the **Batch API**.

This notebook is self-contained (stdlib only) so it runs without depending on PR #9 merging first.

## Pricing and cost model

In [1]:
# USD per 1,000,000 tokens: (input, output).
# Mirrors scripts/usage_tracker.py's MODEL_PRICING (PR #9, cost accounting
# utility) so figures stay consistent with the rest of the repo. Kept as a
# local copy here so this notebook has no dependency on that PR merging.
MODEL_PRICING = {
    "claude-opus-5": (5.00, 25.00),
    "claude-sonnet-5": (3.00, 15.00),
    "claude-haiku-4-5": (1.00, 5.00),
}

CACHE_WRITE_MULTIPLIER = 1.25  # cache_creation_input_tokens, vs input rate
CACHE_READ_MULTIPLIER = 0.1    # cache_read_input_tokens, vs input rate
BATCH_DISCOUNT = 0.5           # Anthropic Batch API: 50% off both rates


def calculate_cost(input_tokens, output_tokens, model,
                    cache_read=0, cache_creation=0, batch=False):
    if model not in MODEL_PRICING:
        raise ValueError(f"no pricing for model '{model}'")
    input_rate, output_rate = MODEL_PRICING[model]
    discount = BATCH_DISCOUNT if batch else 1.0
    cost = input_tokens * input_rate / 1_000_000
    cost += output_tokens * output_rate / 1_000_000
    cost += cache_creation * (input_rate * CACHE_WRITE_MULTIPLIER) / 1_000_000
    cost += cache_read * (input_rate * CACHE_READ_MULTIPLIER) / 1_000_000
    return cost * discount

## Usage log

In [2]:
# Synthetic usage log: one week of calls across WKB-adjacent routes.
# Shape mirrors what UsageTracker.record() sees per call. "batch_ok" marks
# routes that don't need a synchronous reply (safe for the Batch API);
# "min_model" is the cheapest model each route has actually been validated
# against (i.e. output quality holds up on that model for that route).
USAGE_LOG = [
    {"route": "triangulation-orchestrator", "model": "claude-opus-5",
     "calls": 40, "input_tokens": 3200, "output_tokens": 900,
     "cache_read": 0, "cache_creation": 0,
     "batch_ok": False, "min_model": "claude-opus-5"},
    {"route": "wkb-check-validation", "model": "claude-sonnet-5",
     "calls": 500, "input_tokens": 600, "output_tokens": 120,
     "cache_read": 0, "cache_creation": 0,
     "batch_ok": True, "min_model": "claude-haiku-4-5"},
    {"route": "bridge-restore-summary", "model": "claude-sonnet-5",
     "calls": 150, "input_tokens": 2400, "output_tokens": 500,
     "cache_read": 0, "cache_creation": 2400,
     "batch_ok": False, "min_model": "claude-sonnet-5"},
    {"route": "dispatch-topic-classification", "model": "claude-sonnet-5",
     "calls": 800, "input_tokens": 300, "output_tokens": 40,
     "cache_read": 0, "cache_creation": 0,
     "batch_ok": True, "min_model": "claude-haiku-4-5"},
    {"route": "drive-persistence-audit", "model": "claude-opus-5",
     "calls": 20, "input_tokens": 5000, "output_tokens": 1200,
     "cache_read": 0, "cache_creation": 0,
     "batch_ok": True, "min_model": "claude-sonnet-5"},
]

## Baseline cost

In [3]:
def route_baseline_cost(r):
    return calculate_cost(
        r["input_tokens"] * r["calls"],
        r["output_tokens"] * r["calls"],
        r["model"],
        cache_read=r["cache_read"] * r["calls"],
        cache_creation=r["cache_creation"] * r["calls"],
    )


print("=== Baseline cost by route ===")
baseline_by_route = {}
for r in USAGE_LOG:
    c = route_baseline_cost(r)
    baseline_by_route[r["route"]] = c
    print(f"{r['route']:32s} {r['model']:18s} calls={r['calls']:4d}  ${c:8.2f}")

baseline_total = sum(baseline_by_route.values())
print(f"{'TOTAL':51s} ${baseline_total:8.2f}")

=== Baseline cost by route ===
triangulation-orchestrator       claude-opus-5      calls=  40  $    1.54
wkb-check-validation             claude-sonnet-5    calls= 500  $    1.80
bridge-restore-summary           claude-sonnet-5    calls= 150  $    3.56
dispatch-topic-classification    claude-sonnet-5    calls= 800  $    1.20
drive-persistence-audit          claude-opus-5      calls=  20  $    1.10
TOTAL                                               $    9.19


## Optimization levers

In [4]:
# Lever 1: prompt caching on repeated-context routes.
# bridge-restore-summary already writes cache; wkb-check-validation and
# dispatch-topic-classification are high-volume routes with near-identical
# input each call — good caching candidates. Assume 90% of calls after the
# first hit a warm cache instead of paying full input price.
print("=== Lever 1: prompt caching on repeated-context routes ===")
CACHEABLE_ROUTES = {"wkb-check-validation", "dispatch-topic-classification"}
cache_savings = 0.0
for r in USAGE_LOG:
    if r["route"] not in CACHEABLE_ROUTES:
        continue
    calls = r["calls"]
    warm_calls = int(calls * 0.9)
    cold_calls = calls - warm_calls
    optimized = calculate_cost(
        r["input_tokens"] * cold_calls, r["output_tokens"] * calls, r["model"],
        cache_creation=r["input_tokens"] * cold_calls,
    ) + calculate_cost(
        0, 0, r["model"], cache_read=r["input_tokens"] * warm_calls,
    )
    saved = baseline_by_route[r["route"]] - optimized
    cache_savings += saved
    print(f"{r['route']:32s} baseline=${baseline_by_route[r['route']]:.2f}  "
          f"optimized=${optimized:.2f}  saved=${saved:.2f}")
print(f"Lever 1 total savings: ${cache_savings:.2f}")

=== Lever 1: prompt caching on repeated-context routes ===
wkb-check-validation             baseline=$1.80  optimized=$1.18  saved=$0.62
dispatch-topic-classification    baseline=$1.20  optimized=$0.71  saved=$0.49
Lever 1 total savings: $1.11


### Lever 2: model right-sizing

In [5]:
# Lever 2: model right-sizing — route to the cheapest model already
# validated for that route's task, instead of the current default.
print("=== Lever 2: model right-sizing ===")
rightsizing_savings = 0.0
for r in USAGE_LOG:
    if r["min_model"] == r["model"]:
        continue
    downsized = dict(r, model=r["min_model"])
    optimized = route_baseline_cost(downsized)
    saved = baseline_by_route[r["route"]] - optimized
    rightsizing_savings += saved
    print(f"{r['route']:32s} {r['model']} -> {r['min_model']}  "
          f"baseline=${baseline_by_route[r['route']]:.2f}  "
          f"optimized=${optimized:.2f}  saved=${saved:.2f}")
print(f"Lever 2 total savings: ${rightsizing_savings:.2f}")

=== Lever 2: model right-sizing ===
wkb-check-validation             claude-sonnet-5 -> claude-haiku-4-5  baseline=$1.80  optimized=$0.60  saved=$1.20
dispatch-topic-classification    claude-sonnet-5 -> claude-haiku-4-5  baseline=$1.20  optimized=$0.40  saved=$0.80
drive-persistence-audit          claude-opus-5 -> claude-sonnet-5  baseline=$1.10  optimized=$0.66  saved=$0.44
Lever 2 total savings: $2.44


### Lever 3: Batch API

In [6]:
# Lever 3: Anthropic Batch API (50% off both input and output) for any
# route that doesn't need a synchronous reply.
print("=== Lever 3: Batch API (50% off) for batch-eligible routes ===")
batch_savings = 0.0
for r in USAGE_LOG:
    if not r["batch_ok"]:
        continue
    saved = baseline_by_route[r["route"]] * BATCH_DISCOUNT
    batch_savings += saved
    print(f"{r['route']:32s} baseline=${baseline_by_route[r['route']]:.2f}  "
          f"saved=${saved:.2f}")
print(f"Lever 3 total savings: ${batch_savings:.2f}")

=== Lever 3: Batch API (50% off) for batch-eligible routes ===
wkb-check-validation             baseline=$1.80  saved=$0.90
dispatch-topic-classification    baseline=$1.20  saved=$0.60
drive-persistence-audit          baseline=$1.10  saved=$0.55
Lever 3 total savings: $2.05


## Combined savings

In [7]:
# Combined: apply all three levers together per route (right-sized model,
# cache-aware call mix, batch discount where eligible) — the levers are
# independent axes, so they compose rather than being mutually exclusive.
def route_optimized_cost(r):
    model = r["min_model"]
    calls = r["calls"]
    if r["route"] in CACHEABLE_ROUTES:
        warm_calls = int(calls * 0.9)
        cold_calls = calls - warm_calls
        cost = calculate_cost(
            r["input_tokens"] * cold_calls, r["output_tokens"] * calls, model,
            cache_creation=r["input_tokens"] * cold_calls,
            batch=r["batch_ok"],
        ) + calculate_cost(
            0, 0, model, cache_read=r["input_tokens"] * warm_calls,
            batch=r["batch_ok"],
        )
    else:
        cost = calculate_cost(
            r["input_tokens"] * calls, r["output_tokens"] * calls, model,
            cache_read=r["cache_read"] * calls,
            cache_creation=r["cache_creation"] * calls,
            batch=r["batch_ok"],
        )
    return cost


print("=== Combined optimized total ===")
combined_total = 0.0
for r in USAGE_LOG:
    opt = route_optimized_cost(r)
    combined_total += opt
    print(f"{r['route']:32s} baseline=${baseline_by_route[r['route']]:8.2f}  "
          f"optimized=${opt:8.2f}")

print(f"\nBaseline total:  ${baseline_total:.2f}")
print(f"Optimized total: ${combined_total:.2f}")
print(f"Combined savings: ${baseline_total - combined_total:.2f} "
      f"({(baseline_total - combined_total) / baseline_total:.1%})")

=== Combined optimized total ===
triangulation-orchestrator       baseline=$    1.54  optimized=$    1.54
wkb-check-validation             baseline=$    1.80  optimized=$    0.20
bridge-restore-summary           baseline=$    3.56  optimized=$    3.56
dispatch-topic-classification    baseline=$    1.20  optimized=$    0.12
drive-persistence-audit          baseline=$    1.10  optimized=$    0.33

Baseline total:  $9.19
Optimized total: $5.74
Combined savings: $3.45 (37.6%)


## Recommendations

In [8]:
# Recommendations, ranked by savings — generated from the numbers above,
# not hand-typed, so this list can't drift from the calculation.
recommendations = [
    ("Model right-sizing", rightsizing_savings,
     "Route wkb-check-validation and dispatch-topic-classification to "
     "claude-haiku-4-5, and drive-persistence-audit to claude-sonnet-5 — "
     "each was already over-provisioned for its task."),
    ("Batch API", batch_savings,
     "Move batch_ok routes (wkb-check-validation, "
     "dispatch-topic-classification, drive-persistence-audit) off the "
     "synchronous Messages API onto the Batch API; 50% off with no "
     "quality tradeoff since none need a same-second reply."),
    ("Prompt caching", cache_savings,
     "Add a cache_control breakpoint on the stable prefix of "
     "wkb-check-validation and dispatch-topic-classification prompts — "
     "both call with near-identical input at high volume."),
]
recommendations.sort(key=lambda x: x[1], reverse=True)

print("=== Recommendations, ranked by savings ===")
for name, savings, note in recommendations:
    print(f"\n{name}: ${savings:.2f}/week")
    print(f"  {note}")

print(f"\nCombined (all levers together): "
      f"${baseline_total - combined_total:.2f}/week "
      f"({(baseline_total - combined_total) / baseline_total:.1%} of baseline)")

=== Recommendations, ranked by savings ===

Model right-sizing: $2.44/week
  Route wkb-check-validation and dispatch-topic-classification to claude-haiku-4-5, and drive-persistence-audit to claude-sonnet-5 — each was already over-provisioned for its task.

Batch API: $2.05/week
  Move batch_ok routes (wkb-check-validation, dispatch-topic-classification, drive-persistence-audit) off the synchronous Messages API onto the Batch API; 50% off with no quality tradeoff since none need a same-second reply.

Prompt caching: $1.11/week
  Add a cache_control breakpoint on the stable prefix of wkb-check-validation and dispatch-topic-classification prompts — both call with near-identical input at high volume.

Combined (all levers together): $3.45/week (37.6% of baseline)
